<a href="https://colab.research.google.com/github/Jyotsna135-bit/GenerativeAI/blob/main/GenerativeAI/Notebooks/Deployment/Ollama/elora_chatbot_vectordb_documented.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# eLORA RAG Chatbot v2 — Vector Database + Evaluation

The first notebook (`elora_chatbot_documented.ipynb`) built hybrid retrieval manually — looping over all 202 rows in Python, computing cosine similarity and keyword overlap by hand. That works for 202 rows, but it doesn't scale, and it's not how production RAG systems are actually built.

This notebook replaces that manual loop with a **proper vector database** and adds **evaluation** so we can measure how well the system actually performs, not just eyeball a few test queries.

## What's new here
1. **Vector database** — Qdrant, storing both dense (semantic) and sparse (keyword/BM25) vectors for every Q&A pair
2. **Hybrid retrieval done properly** — Qdrant runs both searches internally and merges them with Reciprocal Rank Fusion (RRF), instead of us hand-coding the score combination
3. **Top-k tuning** — testing k = 1, 3, 5, 10 to find what actually works best, instead of guessing
4. **Full evaluation** — running all 202 questions through the pipeline and scoring every generated answer against the ground-truth answer using the LLM itself as a judge

## Why a vector database instead of the manual approach

Looping through all rows in Python and computing similarity by hand works fine at 202 rows, but it doesn't hold up:
- It's O(n) per query — every single question gets compared against every row, every time. With thousands of FAQ entries this gets slow.
- There's no indexing — a real vector database builds an index (HNSW for dense vectors) so it doesn't need to compare against every single point.
- Keyword search done by hand (simple word overlap) is much weaker than BM25, which accounts for term frequency and document length properly.
- A vector database is also what you'd actually use in a real deployment — this notebook is a step closer to a production-shaped system.



## Installing Ollama (for the LLM) and Qdrant + FastEmbed (for the vector DB)

In [ ]:
!apt-get install -y zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh
!pip install "qdrant-client[fastembed]" pandas openpyxl tqdm openai -q

## Starting Ollama and Pulling the LLM

We only need the LLM from Ollama this time — embeddings are handled by Qdrant's FastEmbed integration instead.

In [ ]:
import subprocess, time, requests

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(15):
    try:
        if requests.get("http://localhost:11434").status_code == 200:
            print("ollama is running")
            break
    except:
        time.sleep(1)

!ollama pull qwen3:0.6b

## Step 1 — Load the FAQ Data

Same cleaning as the first notebook — strip HTML entities and normalise stray characters from the raw spreadsheet text.

In [ ]:
import pandas as pd
import re

df = pd.read_excel("eLORA-Jyotsna.xlsx")

def clean_text(text):
    text = str(text)
    text = re.sub(r"&#\d+;", " ", text)
    text = text.replace("?", "'")
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["faq_question"] = df["faq_question"].apply(clean_text)
df["faq_answer"] = df["faq_answer"].apply(clean_text)

print(f"Loaded {len(df)} Q&A pairs")
df.head(3)

## Step 2 — Set Up Qdrant and Build the Collection

We use `QdrantClient(":memory:")` — Qdrant's in-process local mode, no separate server needed, perfect for Colab.

Two embedding models are configured:
- **Dense:** `sentence-transformers/all-MiniLM-L6-v2` — fast, well-tested general-purpose embedding model, captures semantic meaning
- **Sparse:** `Qdrant/bm25` — generates BM25-style sparse vectors for exact keyword matching

Both get computed automatically when we upload documents — FastEmbed handles this under the hood.

In [ ]:
from qdrant_client import QdrantClient, models

DENSE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SPARSE_MODEL = "Qdrant/bm25"
COLLECTION_NAME = "elora_faq"

client = QdrantClient(":memory:")

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "dense": models.VectorParams(
            size=client.get_embedding_size(DENSE_MODEL),
            distance=models.Distance.COSINE,
        )
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams(index=models.SparseIndexParams(on_disk=False))
    },
)

print(f"Collection '{COLLECTION_NAME}' created with dense + sparse vector support")

## Step 3 — Upload the FAQ Data

Each FAQ question gets embedded twice — once as a dense vector, once as a sparse vector — and stored as a point in the collection, with the question, answer, and row index stored as payload (metadata) alongside it.

We embed `faq_question` only (not the answer) since that's what we're matching the user's query against — this mirrors the first notebook's design.

In [ ]:
from tqdm import tqdm

points = []
for idx, row in df.iterrows():
    points.append(
        models.PointStruct(
            id=idx,
            vector={
                "dense": models.Document(text=row["faq_question"], model=DENSE_MODEL),
                "sparse": models.Document(text=row["faq_question"], model=SPARSE_MODEL),
            },
            payload={
                "faq_question": row["faq_question"],
                "faq_answer": row["faq_answer"],
                "row_index": int(idx),
            },
        )
    )

print("uploading and embedding all FAQ entries — this takes a minute or two on first run...")
client.upsert(collection_name=COLLECTION_NAME, points=points)
print(f"Uploaded {len(points)} points to Qdrant")

## Step 4 — Hybrid Retrieval Function

This is the core retrieval call. We use `prefetch` to run both the sparse and dense searches, then combine them with `FusionQuery(fusion=models.Fusion.RRF)` — Reciprocal Rank Fusion. RRF combines two ranked lists by rank position rather than raw score, which avoids the problem of dense (cosine, range -1 to 1) and sparse (BM25, unbounded) scores being on completely different scales.

`top_k` controls how many results come back — we'll tune this in Step 5.

In [ ]:
def hybrid_search(query, top_k=5):
    response = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            models.Prefetch(
                query=models.Document(text=query, model=SPARSE_MODEL),
                using="sparse",
                limit=top_k,
            ),
            models.Prefetch(
                query=models.Document(text=query, model=DENSE_MODEL),
                using="dense",
                limit=top_k,
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=top_k,
    )
    return response.points

# quick sanity check
results = hybrid_search("I forgot my password", top_k=3)
for r in results:
    print(f"score={r.score:.4f}  |  {r.payload['faq_question']}")

## Step 5 — Finding the Right Value of k

`k` is how many candidate FAQ entries we retrieve before generating an answer. There's a real tradeoff:

- **k too small (k=1):** if the single best match happens to be wrong, there's no fallback — the bot confidently answers from the wrong FAQ entry.
- **k too large (k=10):** the prompt gets cluttered with irrelevant entries, increasing the chance the LLM mixes details from multiple FAQs together, or gets distracted by something irrelevant (the article's "don't dump everything into the prompt" pitfall, just at a smaller scale).

We test **recall@k** — for each of the 202 questions, does the correct matching FAQ entry (the one it came from) appear anywhere in the top-k retrieved results? This tells us how much retrieval improves as k grows, and where the improvement starts to flatten out.

In [ ]:
def evaluate_recall_at_k(k_values):
    results = {}
    for k in k_values:
        correct = 0
        for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"k={k}"):
            retrieved = hybrid_search(row["faq_question"], top_k=k)
            retrieved_ids = [r.payload["row_index"] for r in retrieved]
            if idx in retrieved_ids:
                correct += 1
        recall = correct / len(df)
        results[k] = recall
        print(f"k={k}: recall@{k} = {recall:.4f} ({correct}/{len(df)})")
    return results

k_values_to_test = [1, 3, 5, 10]
recall_results = evaluate_recall_at_k(k_values_to_test)

## Step 5b — Visualise and Pick the Best k

Note this test uses each FAQ's own question as the query — it tells us how well retrieval recovers the *exact* source entry, which is the ceiling case. In practice, users paraphrase, so real-world recall will be a bit lower than this. Still useful for picking a sensible k: we want the smallest k where recall is already close to its maximum, since going bigger from there only adds prompt noise without meaningfully improving retrieval.

In [ ]:
import matplotlib.pyplot as plt

ks = list(recall_results.keys())
recalls = list(recall_results.values())

plt.figure(figsize=(7, 4))
plt.plot(ks, recalls, marker="o")
plt.xlabel("k (number of retrieved results)")
plt.ylabel("Recall@k")
plt.title("Retrieval Recall vs k")
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.show()

# pick the smallest k where recall is within 1% of the best recall achieved
best_recall = max(recalls)
BEST_K = min(k for k, r in recall_results.items() if r >= best_recall - 0.01)
print(f"\nChosen k = {BEST_K} (recall@{BEST_K} = {recall_results[BEST_K]:.4f}, best possible = {best_recall:.4f})")

## Step 6 — Generation: Retrieve Top-k, Then Answer

With `BEST_K` decided, this is the full pipeline: retrieve the top-k candidates, pass only those (not the whole dataset) to the LLM, and ask it to answer using only that retrieved context.

We also carry over the confidence check from the first notebook — if the top RRF score is too low, the bot says it isn't confident rather than guessing.

In [ ]:
from openai import OpenAI

llm_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

MIN_RRF_SCORE = 0.01  # RRF scores are small numbers (1/rank based) — tune based on testing

def generate_answer(user_query, k=None, verbose=False):
    k = k or BEST_K
    retrieved = hybrid_search(user_query, top_k=k)

    if not retrieved or retrieved[0].score < MIN_RRF_SCORE:
        return {
            "answer": "I'm not confident this question is covered in the eLORA FAQ. Please check the eLORA help section or contact AERB support directly.",
            "found": False,
            "matched_question": None,
            "top_score": retrieved[0].score if retrieved else 0.0,
        }

    context_blocks = []
    for r in retrieved:
        context_blocks.append(f"Q: {r.payload['faq_question']}\nA: {r.payload['faq_answer']}")
    context = "\n\n".join(context_blocks)

    if verbose:
        print(f"Retrieved {len(retrieved)} candidates, top score = {retrieved[0].score:.4f}")
        print(f"Best match: {retrieved[0].payload['faq_question']}\n")

    prompt = f"""You are a helpful assistant for the eLORA system (AERB's radiation licensing portal).
Use ONLY the FAQ entries below to answer the user's question. Pick the most relevant one — do not mix unrelated entries together, and do not add information that isn't in them.

{context}

User's question: {user_query}

Give a clear, direct answer based only on the FAQ entries above."""

    response = llm_client.chat.completions.create(
        model="qwen3:0.6b",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300
    )

    return {
        "answer": response.choices[0].message.content,
        "found": True,
        "matched_question": retrieved[0].payload["faq_question"],
        "top_score": retrieved[0].score,
    }

# quick test
result = generate_answer("How do I check the status of my application?", verbose=True)
print("Answer:", result["answer"])

## Step 7 — Evaluation: LLM-as-Judge for All 202 Pairs

This is the part that actually tells us how good the system is, rather than relying on a handful of manually tried examples.

For every one of the 202 FAQ questions:
1. Run it through `generate_answer()` — full retrieval + generation pipeline
2. Ask the LLM to judge: does the generated answer convey the same information as the ground-truth answer?
3. The judge returns a score (1-5) and a short reason

We use a **separate, structured judging prompt** rather than reusing the same model call — the judge needs to compare two pieces of text, not generate a fresh answer, so we ask it explicitly for that comparison.

In [ ]:
def judge_answer(question, ground_truth, generated):
    judge_prompt = f"""You are evaluating a chatbot's answer against the correct reference answer.

Question: {question}

Reference (correct) answer: {ground_truth}

Chatbot's answer: {generated}

Score the chatbot's answer from 1 to 5:
5 = fully correct, conveys the same information as the reference
4 = mostly correct, minor details missing or slightly off
3 = partially correct, captures some but not all key information
2 = mostly incorrect, but topic is related
1 = completely wrong or unrelated to the reference answer

Respond in EXACTLY this format, nothing else:
SCORE: <number>
REASON: <one short sentence>"""

    response = llm_client.chat.completions.create(
        model="qwen3:0.6b",
        messages=[{"role": "user", "content": judge_prompt}],
        max_tokens=80,
        temperature=0.0
    )
    return response.choices[0].message.content

In [ ]:
import re

def parse_judge_output(text):
    score_match = re.search(r"SCORE:\s*(\d)", text)
    reason_match = re.search(r"REASON:\s*(.+)", text)
    score = int(score_match.group(1)) if score_match else None
    reason = reason_match.group(1).strip() if reason_match else text.strip()
    return score, reason

eval_records = []

print(f"Evaluating all {len(df)} Q&A pairs — this will take a while...")
for idx, row in tqdm(df.iterrows(), total=len(df)):
    question = row["faq_question"]
    ground_truth = row["faq_answer"]

    result = generate_answer(question)
    generated = result["answer"]

    judge_raw = judge_answer(question, ground_truth, generated)
    score, reason = parse_judge_output(judge_raw)

    eval_records.append({
        "row_index": idx,
        "question": question,
        "ground_truth": ground_truth,
        "generated_answer": generated,
        "found": result["found"],
        "matched_question": result["matched_question"],
        "retrieval_correct": result["matched_question"] == question,
        "judge_score": score,
        "judge_reason": reason,
    })

eval_df = pd.DataFrame(eval_records)
print("\nEvaluation complete.")
eval_df.head()

## Step 8 — Results Summary

This is the outcome that should drive the next development cycle — where the pipeline is strong, and where it's weak.

In [ ]:
valid_scores = eval_df["judge_score"].dropna()

print("=" * 60)
print("OVERALL RESULTS")
print("=" * 60)
print(f"Total questions evaluated: {len(eval_df)}")
print(f"Retrieval found the exact source FAQ: {eval_df['retrieval_correct'].sum()} / {len(eval_df)} ({eval_df['retrieval_correct'].mean()*100:.1f}%)")
print(f"Average judge score (1-5): {valid_scores.mean():.2f}")
print()
print("Score distribution:")
print(eval_df["judge_score"].value_counts().sort_index(ascending=False))
print()
print(f"Answers scored 4 or 5 (good): {(valid_scores >= 4).sum()} ({(valid_scores >= 4).mean()*100:.1f}%)")
print(f"Answers scored 1 or 2 (poor): {(valid_scores <= 2).sum()} ({(valid_scores <= 2).mean()*100:.1f}%)")

In [ ]:
plt.figure(figsize=(7, 4))
eval_df["judge_score"].value_counts().sort_index().plot(kind="bar")
plt.xlabel("Judge Score (1-5)")
plt.ylabel("Number of Questions")
plt.title("Distribution of LLM-Judge Scores Across All 202 Q&A Pairs")
plt.xticks(rotation=0)
plt.grid(True, alpha=0.3, axis="y")
plt.show()

## Step 9 — Looking at the Failures

Averages hide the interesting part. This pulls out the worst-scoring cases so we can actually see what's going wrong — wrong retrieval, the LLM mixing up entries, or something else entirely.

In [ ]:
worst_cases = eval_df[eval_df["judge_score"] <= 2].sort_values("judge_score")

print(f"{len(worst_cases)} low-scoring cases out of {len(eval_df)}\n")

for _, row in worst_cases.head(10).iterrows():
    print("-" * 70)
    print(f"Question: {row['question']}")
    print(f"Retrieval matched correct FAQ: {row['retrieval_correct']}")
    print(f"Ground truth: {row['ground_truth'][:150]}")
    print(f"Generated:    {row['generated_answer'][:150]}")
    print(f"Judge score: {row['judge_score']} — {row['judge_reason']}")
    print()

## Step 10 — Save the Evaluation Results

Saving the full evaluation as a CSV so it can be reviewed outside the notebook, shared, or compared against future iterations of the pipeline.

In [ ]:
eval_df.to_csv("elora_rag_evaluation_results.csv", index=False)
print("Saved to elora_rag_evaluation_results.csv")
print("Download it from the Colab Files panel (left sidebar) to keep a copy.")

## Summary and Next Steps

| | |
|---|---|
| Vector database | Qdrant, in-memory mode |
| Dense embedding model | `sentence-transformers/all-MiniLM-L6-v2` |
| Sparse embedding model | `Qdrant/bm25` |
| Fusion method | Reciprocal Rank Fusion (RRF) |
| Chosen k | see `BEST_K` from Step 5 |
| LLM | `qwen3:0.6b` via Ollama |
| Evaluation method | LLM-as-judge, 1-5 scale, all 202 pairs |

**What this evaluation tells us, and what it doesn't:**
- It measures whether the system can find and correctly answer questions that are worded close to how they appear in the FAQ. It does **not** directly measure how well it handles heavily paraphrased or completely novel questions — that would need a separate test set of paraphrased queries (something worth building next).
- The retrieval recall numbers from Step 5 and the generation scores from Step 7 together tell us where the bottleneck is: if retrieval recall is high but judge scores are low, the problem is in generation (the LLM not using the context well). If retrieval recall is already low, no amount of prompt tuning will fix it — the embedding/fusion setup needs work first.
- `qwen3:0.6b` is being used both as the answer-generator and the judge here. Using the same (small) model for both isn't ideal — a judge model that's different from (and ideally stronger than) the generator would give a more independent evaluation. Worth revisiting if a bigger model becomes available.

**Possible directions for the next development cycle**, depending on what the failure analysis in Step 9 shows:
- If retrieval is the bottleneck — try different embedding models, or weight dense vs. sparse differently instead of plain RRF.
- If generation is the bottleneck — try a slightly larger LLM, or tighten the prompt's instructions.
- Build a second eval set of *paraphrased* questions (not the exact FAQ wording) to test real-world robustness rather than just exact-match recovery.